# T13: Composite Domains

## What You'll Learn

Spindle ships with 13+ standalone domains — retail, HR, healthcare, insurance,
and more. But real enterprises don't live in a single domain. A company has
employees (HR), sells products (retail), and carries liability policies
(insurance) — all at the same time.

**Composite domains** let you merge multiple domain schemas into a single
generation run, producing a unified dataset with prefixed table names and
optional cross-domain relationships.

In this tutorial you will:

1. **Understand** what composite domains are and when to use them
2. **Create** a composite of the Retail and HR domains
3. **Generate** composite data at small scale
4. **Inspect** prefixed table names to see how domains stay organized
5. **Explore** cross-domain relationships
6. **Use** a named preset for common domain combinations

## Why Composite Domains?

| Without Composites | With Composites |
|---|---|
| Generate each domain separately | One call generates everything |
| Manual key alignment across datasets | Shared seed ensures consistency |
| Table name collisions (`customer` in retail vs. insurance) | Prefixed names (`retail_customer`, `insurance_policyholder`) |
| No cross-domain FKs | Optional cross-domain relationships |

## Prerequisites

- `sqllocks-spindle` installed
- Familiarity with at least one Spindle domain (see T01 or T07)

## Time Estimate

**~10 minutes**

In [1]:
# Uncomment the line below if sqllocks-spindle is not yet installed.
# %pip install sqllocks-spindle

print("Installation cell ready. Uncomment the pip line above if needed.")

Installation cell ready. Uncomment the pip line above if needed.


## Step 1 — Create a Composite Domain

**What we're about to do:** Import the `RetailDomain` and `HrDomain` classes,
then wrap them in a `CompositeDomain`. This tells Spindle to generate both
schemas in a single pass.

**Why this matters:** A `CompositeDomain` is not just a convenience wrapper —
it coordinates seed propagation, key ranges, and table prefixing so that the
two domains coexist without collisions. Each domain gets its own key space,
and table names are automatically prefixed with the domain name.

**What to expect:** The composite object will report the combined table count
from both domains.

In [2]:
from sqllocks_spindle import Spindle
from sqllocks_spindle.domains.retail import RetailDomain
from sqllocks_spindle.domains.hr import HrDomain
from sqllocks_spindle.domains.composite import CompositeDomain

retail = RetailDomain()
hr = HrDomain()
composite = CompositeDomain(domains=[retail, hr])

print(f"Composite domain created")
print(f"  Included domains: {[d.name for d in composite.child_domains]}")
print(f"  Combined tables:  {len(composite.get_schema().tables)}")
print(f"\nRetail tables:  {list(retail.get_schema().tables.keys())}")
print(f"HR tables:      {list(hr.get_schema().tables.keys())}")

Composite domain created
  Included domains: ['retail', 'hr']
  Combined tables:  18

Retail tables:  ['customer', 'address', 'product_category', 'product', 'store', 'promotion', 'order', 'order_line', 'return']
HR tables:      ['department', 'position', 'employee', 'compensation', 'performance_review', 'time_off_request', 'training', 'training_enrollment', 'termination']


## Step 2 — Generate the Composite Dataset

**What we're about to do:** Pass the composite domain into `Spindle().generate()`
just like any single domain. Spindle handles the rest — generating each domain's
tables with the shared seed, applying prefixes, and returning a unified result.

**Why this matters:** The API is identical whether you're generating one domain
or ten. The `scale` parameter applies proportionally to each domain, so `small`
gives you a small retail dataset *and* a small HR dataset.

**What to expect:** A result containing tables from both domains, each prefixed
with its domain name (e.g., `retail_customer`, `hr_employee`).

In [3]:
result = Spindle().generate(domain=composite, scale="small", seed=42)

print("Composite Dataset — Retail + HR")
print("=" * 50)
for table_name, df in result.tables.items():
    print(f"  {table_name:<30} {len(df):>8,} rows")

print(f"\nTotal tables: {len(result.tables)}")
print(f"Total rows:   {sum(len(df) for df in result.tables.values()):,}")

Composite Dataset — Retail + HR
  hr_position                          80 rows
  hr_training                         100 rows
  retail_customer                   1,000 rows
  retail_address                    1,500 rows
  retail_product_category              50 rows
  retail_product                      500 rows
  retail_promotion                    300 rows
  retail_store                        150 rows
  hr_department                        30 rows
  hr_employee                         500 rows
  hr_compensation                   1,500 rows
  hr_performance_review             1,250 rows
  hr_termination                       75 rows
  hr_time_off_request               2,500 rows
  hr_training_enrollment            2,000 rows
  retail_order                      5,000 rows
  retail_order_line                12,500 rows
  retail_return                       850 rows

Total tables: 18
Total rows:   29,885


## Step 3 — Inspect Prefixed Table Names

**What we're about to do:** Group the generated tables by their domain prefix
to see how the composite keeps things organized.

**Why this matters:** In a lakehouse or warehouse with dozens of tables, naming
collisions are a real problem. Both retail and HR might have a `customer` or
`address` table. The domain prefix eliminates ambiguity and makes it trivial
to filter tables by domain in your catalog.

**What to expect:** Tables cleanly separated by prefix — `retail_*` and `hr_*`.

In [4]:
# Group tables by domain prefix
domain_groups = {}
for table_name in result.tables:
    prefix = table_name.split("_")[0]
    domain_groups.setdefault(prefix, []).append(table_name)

for domain_prefix, tables in domain_groups.items():
    total_rows = sum(len(result.tables[t]) for t in tables)
    print(f"\n=== {domain_prefix.upper()} Domain ({len(tables)} tables, {total_rows:,} rows) ===")
    for t in sorted(tables):
        df = result.tables[t]
        print(f"  {t:<30} {len(df):>6,} rows  |  cols: {list(df.columns)[:5]}{'...' if len(df.columns) > 5 else ''}")


=== HR Domain (9 tables, 8,035 rows) ===
  hr_compensation                 1,500 rows  |  cols: ['compensation_id', 'employee_id', 'effective_date', 'base_salary', 'bonus_pct']...
  hr_department                      30 rows  |  cols: ['department_id', 'shared_location_retail_store_id', 'department_name', 'cost_center', 'is_active']
  hr_employee                       500 rows  |  cols: ['employee_id', 'department_id', 'position_id', 'shared_person_retail_customer_id', 'first_name']...
  hr_performance_review           1,250 rows  |  cols: ['review_id', 'employee_id', 'reviewer_id', 'review_date', 'review_period_year']...
  hr_position                        80 rows  |  cols: ['position_id', 'position_title', 'pay_grade', 'is_exempt', 'min_salary']...
  hr_termination                     75 rows  |  cols: ['termination_id', 'employee_id', 'termination_date', 'termination_type', 'reason']...
  hr_time_off_request             2,500 rows  |  cols: ['request_id', 'employee_id', 'leave_typ

## Step 4 — Explore Cross-Domain Relationships

**What we're about to do:** Examine how tables from different domains can
relate to each other. For example, retail store employees might reference
HR employee records, or customer service reps in the retail domain might
map to HR's employee table.

**Why this matters:** In a real enterprise, data doesn't live in silos.
The person who fulfills an order (retail) is the same person who has a
salary record (HR). Composite domains can express these cross-domain
foreign keys, enabling realistic end-to-end analytics.

**What to expect:** We'll inspect column names and look for shared key
patterns that bridge the two domains.

In [5]:
# Inspect columns across domains looking for cross-domain key patterns
print("=== Column Inventory by Table ===")
for table_name, df in result.tables.items():
    fk_cols = [c for c in df.columns if c.endswith("_id") and c != df.columns[0]]
    print(f"  {table_name:<30} FK columns: {fk_cols}")

# Show sample data from both domains side by side
retail_tables = [t for t in result.tables if t.startswith("retail_")]
hr_tables = [t for t in result.tables if t.startswith("hr_")]

print(f"\n=== Sample Retail Data ===")
first_retail = retail_tables[0]
print(f"Table: {first_retail}")
print(result.tables[first_retail].head(3).to_string(index=False))

print(f"\n=== Sample HR Data ===")
first_hr = hr_tables[0]
print(f"Table: {first_hr}")
print(result.tables[first_hr].head(3).to_string(index=False))

=== Column Inventory by Table ===
  hr_position                    FK columns: []
  hr_training                    FK columns: []
  retail_customer                FK columns: []
  retail_address                 FK columns: ['customer_id']
  retail_product_category        FK columns: ['parent_category_id']
  retail_product                 FK columns: ['category_id']
  retail_promotion               FK columns: []
  retail_store                   FK columns: []
  hr_department                  FK columns: ['shared_location_retail_store_id']
  hr_employee                    FK columns: ['department_id', 'position_id', 'shared_person_retail_customer_id', 'manager_id']
  hr_compensation                FK columns: ['employee_id']
  hr_performance_review          FK columns: ['employee_id', 'reviewer_id']
  hr_termination                 FK columns: ['employee_id']
  hr_time_off_request            FK columns: ['employee_id']
  hr_training_enrollment         FK columns: ['employee_id', 'traini

## Step 5 — Using Named Presets

**What we're about to do:** Instead of manually assembling domains, use
Spindle's built-in **presets** — named composite configurations for common
enterprise scenarios. We'll list all available presets and then use one.

**Why this matters:** Presets encode best-practice domain combinations with
pre-configured cross-domain relationships. They save you from having to know
which domains pair well and how their keys should align.

**What to expect:** A list of available presets followed by generation using
one of them.

In [6]:
from sqllocks_spindle.presets import list_presets, get_preset

# List all available presets
print("=== Available Presets ===")
for p in list_presets():
    print(f"  {p.name:<20} {p.description}")

# Use a preset to generate a composite dataset
preset = get_preset("enterprise")
print(f"\n=== Preset: {preset.name} ===")
print(f"Description: {preset.description}")
print(f"Domains:     {preset.domains}")

# Build a CompositeDomain from the preset's domain names and shared entities
from sqllocks_spindle.domains.retail import RetailDomain
from sqllocks_spindle.domains.hr import HrDomain
from sqllocks_spindle.domains.financial import FinancialDomain

_domain_classes = {
    "retail": RetailDomain,
    "hr": HrDomain,
    "financial": FinancialDomain,
}
domain_instances = [_domain_classes[name]() for name in preset.domains]
preset_composite = CompositeDomain(
    domains=domain_instances,
    shared_entities=preset.shared_entities,
)

preset_result = Spindle().generate(domain=preset_composite, scale="small", seed=42)
print(f"\nGenerated {len(preset_result.tables)} tables, {sum(len(df) for df in preset_result.tables.values()):,} total rows")
for table_name, df in preset_result.tables.items():
    print(f"  {table_name:<30} {len(df):>8,} rows")

=== Available Presets ===
  enterprise           Enterprise dataset combining retail, HR, and financial domains
  healthcare_system    Healthcare system with insurance and HR
  smart_factory        Smart factory combining manufacturing, IoT, and supply chain
  digital_commerce     Digital commerce with retail, marketing, and financial data
  campus               University campus combining education and HR
  telecom_bundle       Telecom provider with marketing and billing

=== Preset: enterprise ===
Description: Enterprise dataset combining retail, HR, and financial domains
Domains:     ['retail', 'hr', 'financial']



Generated 28 tables, 63,685 total rows
  financial_branch                    200 rows
  financial_customer                1,000 rows
  financial_loan                      400 rows
  financial_loan_payment            4,800 rows
  financial_transaction_category       40 rows
  hr_department                        30 rows
  hr_position                          80 rows
  hr_employee                         500 rows
  financial_account                 2,200 rows
  financial_card                    1,760 rows
  financial_statement              13,200 rows
  financial_transaction            10,000 rows
  financial_fraud_flag                200 rows
  hr_compensation                   1,500 rows
  hr_performance_review             1,250 rows
  hr_termination                       75 rows
  hr_time_off_request               2,500 rows
  hr_training                         100 rows
  hr_training_enrollment            2,000 rows
  retail_customer                   1,000 rows
  retail_address    

## What's Next?

You've learned how to combine multiple Spindle domains into a single composite
dataset — the foundation for realistic enterprise-scale data generation.

| Notebook | What You'll Learn |
|----------|-------------------|
| **T14: Day 2 — Incremental** | Generate CDC-style deltas on top of your composite dataset |
| **F09: Cross-Domain Enterprise** | Build a full enterprise lakehouse with 3 domains at medium scale |
| **T06: Star Schema Export** | Transform composite data into star schemas with Parquet export |
| **T07: Domain Quickstarts** | Explore each domain individually before combining them |

**Key takeaways:**
- `CompositeDomain` merges multiple domains into a single generation call
- Table names are automatically prefixed (`retail_customer`, `hr_employee`) to avoid collisions
- Cross-domain foreign keys enable realistic enterprise analytics
- Named presets provide ready-made domain combinations for common scenarios
- The API is identical — `Spindle().generate()` works the same for single or composite domains